## Parameters

In [34]:
Business_Category='Home'
Report_Name='ITS Home - FACETS RF Input Detail Control'
Jurisdiction="DC"

StatementMeta(, 05ee7026-595b-4280-8e80-6fa4c3450632, 36, Finished, Available, Finished, False)

## Table Selection

In [35]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.functions import sum as _sum, count, col, array, lit,concat,lpad

df_t = spark.sql("SELECT * FROM ITS_Home.silver.ITS_Home_Report_Tables")
display(df_t)
df_table = df_t.filter((df_t["Report"] == Report_Name))  \
                .select("Table Name")
display(df_table)
Table_Name = df_table.first()["Table Name"]
print(Table_Name)


StatementMeta(, 05ee7026-595b-4280-8e80-6fa4c3450632, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dbaad901-9adf-4a51-97c8-36587d31c80b)

SynapseWidget(Synapse.DataFrame, e3ba56aa-c5e9-489b-b873-0e34aaec3407)

ITS_Home.gold.rf_detailed


## Count of Rows Column

In [36]:
row = df_t.filter(col("Report") == Report_Name).select("Count_Field").first()

if row:
    # 2. Wrap it in double quotes
    count_field = f"{row[0]}"
display(count_field)


StatementMeta(, 05ee7026-595b-4280-8e80-6fa4c3450632, 38, Finished, Available, Finished, False)

'SCCF NUMBER'

## Sort Columns

In [37]:
sort_columns = df_t.filter(col("Report") == Report_Name) \
                .select(array("Sort_Date", "Count_Field")) \
                .rdd.flatMap(lambda x: x[0]) \
                .collect()

print(sort_columns)

StatementMeta(, 05ee7026-595b-4280-8e80-6fa4c3450632, 39, Finished, Available, Finished, False)

['RF EDIT DATE', 'SCCF NUMBER']


## File Name Generation

In [38]:
from pyspark.sql.functions import col,concat,lit,when,last_day, add_months, current_date, date_format,concat,current_timestamp,regexp_extract,to_date

df_par = spark.sql("SELECT * FROM ITS_Home.silver.ITS_Home_Report_List ")
display(df_par)
df_par = df_par.filter( (df_par["Report"] == Report_Name) & (df_par["Jurisdiction"] == Jurisdiction) & (df_par["Business Category"] == Business_Category)) \
                .select("filename")
display(df_par)

df_with_date = df_par.withColumn(
    "PrevMonthEndDate", 
    date_format(last_day(add_months(current_date(), -1)), "yyyyMMdd")
)
distinct_date = df_with_date.select(
    date_format(to_date(col("PrevMonthEndDate"), "yyyyMMdd"), "MMM_yyyy").alias("formatted_date")
).distinct().rdd.flatMap(lambda x: x).collect() # flatMap and collect efficiently create a list .distinct().rdd.flatMap(lambda x: x).collect()

File_Date = str(distinct_date).strip("'[]'")
print(File_Date) 
display(distinct_date)
#current  date for postfix

df_today= df_with_date.withColumn("todaysdate", date_format(current_date(), "yyyyMMdd"))

from pyspark.sql.functions import current_date, add_months, date_format

# Calculate current month and format it
month_str = spark.range(1).select(
    date_format(current_date(), "MMM_yyyy/")).collect()[0][0]

print(month_str)

df_result = df_today.withColumn("file_name", concat(lit('Extracts/'),lit(month_str),col("filename"),col("PrevMonthEndDate"),lit("_"),col("todaysdate")))

Report= df_result.drop('filename', 'PrevMonthEndDate', 'todaysdate','current_month_year')

display(Report)

file_name = Report.first()["file_name"]
print(file_name)
output_path = f"/lakehouse/default/Files/{file_name}.xlsx"
print(output_path)



StatementMeta(, 05ee7026-595b-4280-8e80-6fa4c3450632, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 30cec8d0-d754-4e55-a0fb-d6e55e4b7b21)

SynapseWidget(Synapse.DataFrame, 4ba08f41-ec2c-4b69-bd0d-4a0ba6e31b8b)

Mar_2026


SynapseWidget(Synapse.DataFrame, 018f444f-ae74-4bc2-afa8-9e078a4f416d)

Apr_2026/


SynapseWidget(Synapse.DataFrame, 01a4450e-ab1c-42f6-ac86-30680750cc7d)

Extracts/Apr_2026/Home_RF_CNTL_DETAIL_DC080_212183_213336_20260331_20260414
/lakehouse/default/Files/Extracts/Apr_2026/Home_RF_CNTL_DETAIL_DC080_212183_213336_20260331_20260414.xlsx


## Monthly Folder

In [39]:
# Ensure month_str is defined, e.g., month_str = "2024-03"
from notebookutils import mssparkutils # Preferred over mssparkutils [2]

# Define path relative to Lakehouse Files
folder_path = f"Files/Extracts/{month_str}" 
print(f"Checking path: {folder_path}")

# Create directory if it doesn't exist (handles parents)
mssparkutils.fs.mkdirs(folder_path)

# Verify
if mssparkutils.fs.exists(folder_path):
    print(f"Directory {folder_path} is ready (created or already existed).")
else:
    print(f"Failed to create directory {folder_path}.")


StatementMeta(, 05ee7026-595b-4280-8e80-6fa4c3450632, 41, Finished, Available, Finished, False)

Checking path: Files/Extracts/Apr_2026/
Directory Files/Extracts/Apr_2026/ is ready (created or already existed).


## Code for Table/View to excel 

In [40]:
from pyspark.sql import functions as F
from pyspark.sql import Row
import pandas as pd
import numpy as np
from pyspark.sql.types import DecimalType,DateType,StringType,FloatType,DoubleType
from pyspark.sql.functions import sum as _sum, count, col, array, lit,concat,lpad

# 1. Load Lakehouse table into a Spark DataFrame
#df = spark.read.table(Table_Name).filter("Jurisdiction = 'MD'").orderBy("FACETS STATUS DATE","ITS SCCF NBR")
df = spark.read.table(Table_Name).filter(col("Jurisdiction") == Jurisdiction).orderBy(*sort_columns)

#2. Drop Jurisdiction Column
df = df.drop("Jurisdiction", "Business Category")

print("Table Read")

#3. Get Decimal and Float Type Column List
def get_measure_columns(df):

  decimal_columns = [
      col_name for col_name, dtype in df.dtypes if dtype.startswith('decimal') or dtype.startswith('double')
  ]
  return decimal_columns

measure_cols = get_measure_columns(df)

display(measure_cols)
# 3.1 Define Measure Columns (to be formatted as currency)
def get_decimal_columns(df):

  decimal_columns = [
      col_name for col_name, dtype in df.dtypes if dtype.startswith('decimal')
  ]
  return decimal_columns

decimal_columns = get_decimal_columns(df)
print(decimal_columns)

# 3.2 convert null values to zero 
df= df.fillna(0, subset=measure_cols)

# 3.3 Define Date Type  Columns
def get_date_columns(df) -> list:
    """
    Finds columns in a PySpark DataFrame that are of date or timestamp type.
    """
    # dtypes returns a list of (column_name, data_type) tuples
    dt_cols = [
        col_name for col_name, data_type in df.dtypes 
        if 'date' in data_type or 'timestamp' in data_type
    ]
    return dt_cols
date_cols  = get_date_columns(df)
print(date_cols)

# 3.3.1  Date Type  Columns casting
for date_col in date_cols:
    df = df.withColumn(date_col, col(date_col).cast("date"))

# 4 Create a list of aggregation expressions for decimals
agg_expressions = [F.sum(F.col(c)).alias(c) for c in measure_cols]

# 5 Define Count Rows  Column and it to the aggregate list

count_column = count_field # Replace with your actual column name

agg_expressions.append(concat(lit("Total Rows: "),F.format_number(F.count(F.col(count_column)), 0)).alias(count_field))


# 5.2 Perform the aggregation to get a new DataFrame with a single row of totals
totals_df = df.agg(*agg_expressions)
display(totals_df)

#5.3 Union the Detail and Total dataframes
df_spark = df.unionByName(totals_df,allowMissingColumns=True)

display(df_spark)

for column in decimal_columns:
    df_spark = df_spark.withColumn(column, col(column).cast(DoubleType()))

# 6. Convert to Pandas DataFrame for Excel writing
pdf = df_spark.toPandas()


# 7. Save to Excel with XlsxWriter
output_path = f"/lakehouse/default/Files/{file_name}.xlsx"
print(output_path)
excel_date_format = 'mm/dd/yyyy' 

writer = pd.ExcelWriter(
    output_path,
    engine='xlsxwriter',
    date_format=excel_date_format,
    engine_kwargs={
        'options': {
            'strings_to_numbers': False,                
            'nan_inf_to_errors': True
        }
    }
)

pdf.to_excel(writer, index=False, sheet_name='Sheet1')

workbook  = writer.book
worksheet = writer.sheets['Sheet1']
# 8. --- FORMATTING columns using functions ---
last_row = len(pdf)


#date_format=workbook.add_format({
#    'num_format': '[$-en-US,1]mm/dd/yyyy','align': 'left'})

# 8.1 Define Currency Format (Right Aligned)
currency_format = workbook.add_format({
    'num_format': '[$-en-US,1]$#,##0.00;($#,##0.00)',
    'align': 'right'
})

# 8.2 Define left align Format

left_fmt = workbook.add_format({
    'align': 'left'
})
text_format = workbook.add_format({'num_format': '@'})

# 8.3 Define Header Format
header_format = workbook.add_format({
    'align': 'left',    # Left align
    'bold': True,
    'text_wrap': False,
    'valign': 'top',
    'border': 1
})

bold_fmt = workbook.add_format({'bold': True})
    
# 8.4 Apply Formats and Dynamic Column Sizing
for i, col in enumerate(pdf.columns):
    # Determine column width based on max string length of data or header
    column_len = pdf[col].astype(str).str.len().max()
    column_len = max(column_len, len(col)) + 2 # Add padding

# Apply currency format to measures, others default
    if col in measure_cols:
        worksheet.set_column(i, i, column_len, currency_format)
    else:
        worksheet.set_column(i, i, column_len,left_fmt)

# 8.5 Re-apply header format
for col_num, value in enumerate(pdf.columns.values):
    worksheet.write(0, col_num, value,header_format)

#for col_name in date_cols:
    # Get column index
#    col_idx = pdf.columns.get_loc(col_name)
    # Apply format to the column (width 15 is reasonable for dates)
#    worksheet.set_column(col_idx, col_idx, 15, date_format)

    # Apply format to entire range: (0, 0, nrows, ncols-1)
# 4. Set height for all rows (starting at 1 to skip header)
# Row 0 is the header. Data rows start at 1.
for row_num in range(1, len(pdf) + 1):
    worksheet.set_row(row_num, 36) # 36 points = 48 pixels

writer.close()
# Suceess note
print(f"Exported successfully to {output_path}")



StatementMeta(, 05ee7026-595b-4280-8e80-6fa4c3450632, 42, Finished, Available, Finished, False)

Table Read


SynapseWidget(Synapse.DataFrame, c620082a-5b1a-4217-b580-2e5bfdc69dd0)

['RF TOTAL PAID AMOUNT', 'APPROVED PAYMENT', 'CLAIM LIABILITY AMOUNT', '212183 GL CLAIM LIABILITY AMOUNT', '212183 VARIANCE CLAIM LIABILITY AMOUNT', 'NET LIAB', '213336 GL NET LIABILITY AMOUNT', '213336 VARIANCE NET LIABILITY AMOUNT', 'ACCESS FEE', '552106 GL ACCESS FEE AMT', '552106 VARIANCE ACCESS FEE AMOUNT', 'ADMIN EXP', '796500 GL ADMIN (AEA) AMOUNT', '796500 VARIANCE ADMIN (AEA) AMOUNT']
['RF EDIT DATE', 'RF PAID DATE', 'ADMISSION DATE', 'CFA PROCESSING DATE', 'CFA DEFAULT PAYMENT DATE', 'CFA DISPOSITION DATE', 'BATCH CREATION DATE', 'RECEIPT DATE', 'HOME AUTH DENIAL DATE', 'RF POSTING DATE']


SynapseWidget(Synapse.DataFrame, 8ef76e9d-5dd8-410d-a108-6d89d61aea40)

SynapseWidget(Synapse.DataFrame, 456b6095-1475-4a68-a32e-e6daf8d60772)

/lakehouse/default/Files/Extracts/Apr_2026/Home_RF_CNTL_DETAIL_DC080_212183_213336_20260331_20260414.xlsx
Exported successfully to /lakehouse/default/Files/Extracts/Apr_2026/Home_RF_CNTL_DETAIL_DC080_212183_213336_20260331_20260414.xlsx


 ## Edits on saved Excel File

In [44]:
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.styles import numbers
from openpyxl.styles import Alignment
from datetime import date


# 1. Path to your Lakehouse file
file_path = f"/lakehouse/default/Files/{file_name}.xlsx"
print(file_path)

# Example usage:
# 2. Load workbook with keep_styles=True
wb = load_workbook(file_path)
ws = wb.active

# 3. Get the last row number
last_row = ws.max_row

# 4. Iterate and bold cells in the last row
for cell in ws[last_row]:
    cell.font = Font(bold=True)

# 5. Save the workbook
wb.save(file_path)


StatementMeta(, 05ee7026-595b-4280-8e80-6fa4c3450632, 46, Finished, Available, Finished, False)

/lakehouse/default/Files/Extracts/Apr_2026/Home_RF_CNTL_DETAIL_DC080_212183_213336_20260331_20260414.xlsx


BadZipFile: File is not a zip file